# ⚽ eFootball AI Assistant - End-to-End Colab Training & Benchmark Pipeline

Pipeline tự động huấn luyện và đóng gói mô hình AI nhận diện thời gian thực (Real-time Vision AI) cho **eFootball Mobile**:
- **Mô hình 1: Field Detector** (Bóng, Cầu thủ, Thủ môn, Trọng tài, Con trỏ)
- **Mô hình 2: Pitch Calibration Keypoints** (32 điểm mốc sân bóng để tính ma trận phối cảnh 2D)
- **Mô hình 3: Minimap / Radar** (Nhận diện vị trí toàn bộ 22 cầu thủ trên bản đồ thu nhỏ)
- **Video Tracking Benchmark**: Kiểm thử tracking bóng trên video thực tế, đo FPS và Latency (ms)
- **Export Android Bundle**: Xuất mô hình ONNX / TFLite tối ưu độ trễ thấp để nạp vào Android APK.

## 1. Kết nối Google Drive & Kiểm tra GPU

In [ ]:
from google.colab import drive
import os, sys, shutil, subprocess
from pathlib import Path

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Kiểm tra GPU
import torch
if not torch.cuda.is_available():
    raise SystemError('Vui lòng bật GPU trong Runtime -> Change runtime type -> T4 GPU!')
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'✅ PyTorch Version: {torch.__version__}')

## 2. Thiết lập Môi trường & Clone Repository efootphy

In [ ]:
# 3. Clone Repository efootphy (nhánh upgrade/vision-fusion-qc)
WORKSPACE = Path('/content/efootphy')
if not WORKSPACE.exists():
    !git clone -b upgrade/vision-fusion-qc https://github.com/clonephy-blip/efootphy.git {WORKSPACE}
else:
    !git -C {WORKSPACE} pull

%cd {WORKSPACE}

# 4. Cài đặt các thư viện tự động
!pip install -q ultralytics onnx onnxruntime-gpu opencv-python-headless

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

import ultralytics
print(f'✅ Đã cài đặt thành công Ultralytics v{ultralytics.__version__}!')

## 3. Tự động Tìm kiếm và Kết nối Thư mục Dữ liệu train ai

In [ ]:
# Tự động tìm kiếm thư mục dữ liệu 'train ai' trong Google Drive
DATASET_LOCAL = Path('/content/datasets')
DATASET_LOCAL.mkdir(parents=True, exist_ok=True)
source_folder = None

# Danh sách các đường dẫn ưu tiên kiểm tra trong MyDrive
potential_paths = [
    Path('/content/drive/MyDrive/train ai'),
    Path('/content/drive/MyDrive/train_ai'),
    Path('/content/drive/MyDrive/efootball_detection'),
]

for p in potential_paths:
    if p.exists() and ((p / 'efootball_detection').exists() or p.name == 'efootball_detection'):
        source_folder = p if (p / 'efootball_detection').exists() else p.parent
        break

# Nếu chưa thấy ở vị trí gốc, quét đệ quy MyDrive (kể cả các shortcut / lối tắt)
if source_folder is None and os.path.exists('/content/drive/MyDrive'):
    for root, dirs, files in os.walk('/content/drive/MyDrive', followlinks=True):
        if 'efootball_detection' in dirs or os.path.basename(root) == 'train ai':
            candidate = Path(root)
            if (candidate / 'efootball_detection').exists():
                source_folder = candidate
                break
            elif candidate.name == 'train ai':
                source_folder = candidate
                break

if source_folder and (source_folder / 'efootball_detection').exists():
    print(f'✅ ĐÃ TÌM THẤY THƯ MỤC DỮ LIỆU: {source_folder}')
    print('Danh sách tệp/thư mục trong dataset:')
    for item in sorted(source_folder.iterdir()):
        print(f'  - {item.name}')
else:
    print('⚠️ CHƯA TÌM THẤY THƯ MỤC DỮ LIỆU TRONG MYDRIVE!')
    if os.path.exists('/content/drive/MyDrive'):
        print('Các thư mục hiện có trong MyDrive của bạn:', os.listdir('/content/drive/MyDrive'))
    print('\n👉 HƯỚNG DẪN 3 GIÂY ĐỂ KẾT NỐI DỮ LIỆU:')
    print('1. Chuyển sang Tab "train ai - Google Drive" đang mở sẵn trên trình duyệt Cốc Cốc của bạn.')
    print('2. Bấm vào chữ "train ai" ở đầu trang (cạnh biểu tượng thư mục) -> Chọn "Thêm lối tắt vào Drive" (Add shortcut to Drive).')
    print('3. Chọn "Drive của tôi" (My Drive) rồi bấm "Thêm" (Add).')
    print('4. Quay lại Colab và bấm ▶ Chạy tất cả là hoàn thành ngay!\n')
    raise FileNotFoundError('Vui lòng tạo lối tắt cho thư mục "train ai" vào "Drive của tôi" theo hướng dẫn ở trên!')

In [ ]:
# Chuẩn bị đường dẫn các dataset vào SSD của Colab để train tốc độ cao nhất
def get_dataset_path(folder_name):
    p = source_folder / folder_name
    p_local = DATASET_LOCAL / folder_name
    if p_local.exists() and (p_local / 'data.yaml').exists():
        return p_local
    if p.exists() and (p / 'data.yaml').exists():
        if not p_local.exists():
            print(f'Đang sao chép {folder_name} sang Colab SSD để huấn luyện tốc độ cao...')
            shutil.copytree(p, p_local)
        return p_local
    return None

field_dir = get_dataset_path('efootball_detection')
points_dir = get_dataset_path('efootball-analytics-points.v2i.yolov8')
minimap_dir = get_dataset_path('UI Mini Map')

print(f'Field Dataset: {field_dir}')
print(f'Points Dataset: {points_dir}')
print(f'Minimap Dataset: {minimap_dir}')

## 4. Huấn luyện Mô hình 1: Field Detector (Bóng + Cầu thủ + Trọng tài)

Sử dụng kiến trúc **YOLO11n (Nano)** siêu nhẹ để tối ưu độ trễ (latency < 10ms trên điện thoại).
- 7 Class: `['ball', 'cursor', 'cursor_secondary', 'goalkeeper', 'player', 'powerbar', 'referee']`
- Kích thước ảnh: `320x320` cho ứng dụng mobile real-time.

In [ ]:
from ultralytics import YOLO

if field_dir and (field_dir / 'data.yaml').exists():
    field_data_yaml = field_dir / 'data.yaml'
    field_model = YOLO('yolo11n.pt')

    print('🚀 Bắt đầu huấn luyện mô hình Field (Bóng + Cầu thủ + Trọng tài)...')
    field_results = field_model.train(
        data=str(field_data_yaml),
        epochs=30,
        imgsz=320,          # 320px giúp đạt tốc độ 60+ FPS trên mobile
        batch=16,
        device=0,
        workers=4,
        patience=8,
        save=True,
        project='/content/runs/field',
        name='yolo11n_field_320',
        exist_ok=True
    )

    # Xuất sang định dạng ONNX
    field_onnx_path = field_model.export(format='onnx', imgsz=320, dynamic=False, opset=12)
    print(f'✅ Đã xuất mô hình Field sang ONNX: {field_onnx_path}')
else:
    print('❌ Không tìm thấy dataset Field.')

## 5. Huấn luyện Mô hình 2: Pitch Calibration Keypoints (32 mốc sân)

Mô hình Pose ước lượng 32 điểm nút trên mặt sân cỏ (góc sân, vòng tròn giữa sân, vạch 16m50). Tọa độ này kết hợp với module `Projection.kt` trong app để tính ma trận **Homography**, chuyển đổi mọi vị trí cầu thủ sang bản đồ 2D chiến thuật.

In [ ]:
if points_dir and (points_dir / 'data.yaml').exists():
    points_yaml = points_dir / 'data.yaml'
    pose_model = YOLO('yolo11n-pose.pt')

    print('🚀 Bắt đầu huấn luyện mô hình Sân bóng (32 Keypoints)...')
    pose_results = pose_model.train(
        data=str(points_yaml),
        epochs=30,
        imgsz=640,
        batch=8,
        device=0,
        workers=4,
        patience=8,
        save=True,
        project='/content/runs/pitch',
        name='yolo11n_pose_pitch',
        exist_ok=True
    )
    pose_onnx_path = pose_model.export(format='onnx', imgsz=640, dynamic=False, opset=12)
    print(f'✅ Đã xuất mô hình Pitch Pose sang ONNX: {pose_onnx_path}')
else:
    print('⚠️ Bỏ qua phần Pose do chưa có dataset.')

## 6. Huấn luyện Mô hình 3: UI Minimap Radar

Nhận diện 3 class trên minimap: `ball`, `controlled` (cầu thủ đang điều khiển), `dot` (các cầu thủ khác).

In [ ]:
if minimap_dir and (minimap_dir / 'data.yaml').exists():
    minimap_yaml = minimap_dir / 'data.yaml'
    minimap_model = YOLO('yolo11n.pt')

    print('🚀 Bắt đầu huấn luyện mô hình Minimap Radar...')
    minimap_results = minimap_model.train(
        data=str(minimap_yaml),
        epochs=30,
        imgsz=320,
        batch=16,
        device=0,
        workers=4,
        patience=8,
        save=True,
        project='/content/runs/minimap',
        name='yolo11n_minimap_320',
        exist_ok=True
    )
    minimap_onnx_path = minimap_model.export(format='onnx', imgsz=320, dynamic=False, opset=12)
    print(f'✅ Đã xuất mô hình Minimap sang ONNX: {minimap_onnx_path}')
else:
    print('⚠️ Bỏ qua phần Minimap do chưa có dataset.')

## 7. Video Tracking Benchmark & Kiểm định Độ trễ Realtime

Chạy thử nghiệm tracking trên video trận đấu từ Drive (`MATCH_01_PART_005.mp4`), đo độ trễ trên từng frame và xuất video preview kèm bounding box + quỹ đạo bóng.

In [ ]:
import cv2
import time
import numpy as np

# Tìm file video trận đấu từ source_folder
test_video = None
for vid_name in ['MATCH_01_PART_005.mp4', 'MATCH_01_PART_007.mp4', 'MATCH_01_PART_010.mp4']:
    candidate = source_folder / vid_name
    if candidate.exists():
        test_video = candidate
        break

best_field_model = Path('/content/runs/field/yolo11n_field_320/weights/best.pt')
if test_video and best_field_model.exists():
    print(f'🎬 Đang benchmark video: {test_video.name}')
    model = YOLO(str(best_field_model))
    cap = cv2.VideoCapture(str(test_video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    out_path = '/content/benchmark_replay_preview.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(out_path, fourcc, fps, (width, height))
    
    latencies = []
    ball_history = []
    frame_count = 0
    max_frames = 300  # Benchmark 300 frames (~10s)
    
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret: break
        
        t0 = time.perf_counter()
        # Nâng lên 640px và hạ conf=0.08 để bắt trọn quả bóng nhỏ (tránh bị nuốt pixel khi nén)
        results = model.predict(frame, imgsz=640, conf=0.08, verbose=False, device=0)
        t1 = time.perf_counter()
        
        latencies.append((t1 - t0) * 1000)
        frame_count += 1
        
        # Trích xuất và vẽ kết quả lên video
        res = results[0]
        for box in res.boxes:
            cls_id = int(box.cls[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls_name = model.names[cls_id]
            
            if cls_name == 'ball':
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                ball_history.append((cx, cy))
                cv2.circle(frame, (cx, cy), 8, (0, 0, 255), -1)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            elif 'player' in cls_name or 'goalkeeper' in cls_name:
                color = (0, 255, 0) if cls_name == 'player' else (255, 165, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        
        # Vẽ vệt quỹ đạo bóng
        for i in range(1, len(ball_history[-20:])):
            cv2.line(frame, ball_history[-20:][i - 1], ball_history[-20:][i], (0, 255, 255), 2)
        
        # Vẽ thông số FPS & Latency
        curr_latency = latencies[-1]
        cv2.putText(frame, f'Latency: {curr_latency:.1f}ms | FPS: {1000/curr_latency:.1f}', (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        writer.write(frame)
    
    cap.release()
    writer.release()
    try:
        shutil.copy(out_path, '/content/drive/MyDrive/benchmark_replay_preview.mp4')
        print('💾 Đã tự động lưu video vào Google Drive của bạn (MyDrive/benchmark_replay_preview.mp4)')
    except Exception: pass
    
    avg_latency = np.mean(latencies)
    print(f'\n📊 KẾT QUẢ HIỆU NĂNG:')
    print(f'   - Độ trễ trung bình: {avg_latency:.2f} ms')
    print(f'   - Tốc độ xử lý: ~{1000/avg_latency:.1f} FPS')
    print(f'   - Video demo kết quả: {out_path}')
else:
    print('⚠️ Bỏ qua benchmark video do chưa có video hoặc model chưa train xong.')

## 8. Đóng gói Model Bundle và Lưu ngược về Google Drive cho Android

In [ ]:
import hashlib, json, shutil
from pathlib import Path

# Tìm model minimap
minimap_onnx = Path('/content/runs/minimap/yolo11n_minimap_320/weights/best.onnx')
if not minimap_onnx.exists():
    minimap_onnx = Path('/content/runs/field/yolo11n_field_320/weights/best.onnx')

bundle_dir = Path('/content/android_model_bundle')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)

# 1. Đặt tên đúng chuẩn app Android: model.onnx
target_model = bundle_dir / 'model.onnx'
shutil.copyfile(minimap_onnx, target_model)

# 2. Tính mã băm SHA256
with open(target_model, 'rb') as f:
    model_sha256 = hashlib.sha256(f.read()).hexdigest()

# 3. Tạo manifest.json đúng chuẩn 100% của app Android
manifest = {
    'schema_version': 1,
    'profile': 'minimap',
    'task': 'detect',
    'input_size': 320,
    'input_layout': 'NCHW',
    'output_layout': 'channels_first',
    'normalization': 'RGB/255',
    'classes': ['ball', 'controlled', 'dot'],
    'model_sha256': model_sha256
}

with open(bundle_dir / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

# 4. Đóng gói zip chuẩn chỉ có model.onnx và manifest.json ở thư mục gốc
shutil.make_archive('/content/efootphy_minimap_bundle', 'zip', str(bundle_dir))

# 5. Lưu vào Drive
shutil.copyfile('/content/efootphy_minimap_bundle.zip', '/content/drive/MyDrive/efootphy_minimap_bundle.zip')
print('🎉 ĐÃ TẠO GÓI CHUẨN THÀNH CÔNG: efootphy_minimap_bundle.zip!')

# Tự động tải xuống máy
from google.colab import files
files.download('/content/efootphy_minimap_bundle.zip')